# Optimisation de portefeuille : Markowitz, clustering et backtest sur le CAC 40

**Question** : peut-on construire un portefeuille d'actions françaises plus efficace (meilleur rapport rendement/risque) que l'indice CAC 40 ou qu'une simple répartition égale ?

**Démarche**
1. Données : ~30 grandes actions du CAC 40, 2015 à 2025 (Yahoo Finance, gratuit).
2. Théorie de Markowitz : frontière efficiente, portefeuille de variance minimale, portefeuille de Sharpe maximal.
3. Clustering : regrouper les actions qui évoluent ensemble pour mieux diversifier (CAH, K-means, HRP).
4. Backtest honnête (walk-forward) : rebalancement mensuel, uniquement avec les données passées, frais de transaction inclus.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram
from src import data, markowitz, clustering, backtest

plt.rcParams.update({"figure.figsize": (11, 5), "axes.grid": True, "grid.alpha": 0.3})
DEMO = False   # True = données simulées (si pas d'accès internet). Laisse False pour les vrais résultats.

## 1. Données

In [ ]:
prices, bench = data.load_prices(demo=DEMO)
returns = data.simple_returns(prices)
names = {t: data.TICKERS.get(t, t) for t in prices.columns}
print(f"{prices.shape[1]} actions, {len(prices)} jours, du {prices.index[0]:%d/%m/%Y} au {prices.index[-1]:%d/%m/%Y}")
prices.tail(3)

In [ ]:
mu, sigma = markowitz.estimate(returns)
stats = pd.DataFrame({"Rendement annuel": mu, "Volatilité": np.sqrt(np.diag(sigma))}, index=prices.columns)
stats.index = stats.index.map(names)
ax = stats.plot.scatter("Volatilité", "Rendement annuel", s=40)
for n, r in stats.iterrows():
    ax.annotate(n, (r["Volatilité"], r["Rendement annuel"]), fontsize=7, alpha=0.8)
ax.set_title("Chaque action : rendement vs risque (sur toute la période)");

**Corrélations.** La diversification marche d'autant mieux que les actions sont peu corrélées. La heatmap, réordonnée par clustering, fait apparaître des blocs (banques, luxe, industrie...).

In [ ]:
labels_h, Z = clustering.hierarchical_clusters(returns, n_clusters=6)
order = labels_h.sort_values().index
corr = returns[order].corr()
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(order)), [names[t] for t in order], rotation=90, fontsize=7)
ax.set_yticks(range(len(order)), [names[t] for t in order], fontsize=7)
plt.colorbar(im, fraction=0.046); ax.set_title("Matrice de corrélation (ordonnée par cluster)"); ax.grid(False);

## 2. Markowitz : la frontière efficiente

Pour des poids $w$, le rendement espéré est $w^\top \mu$ et la variance $w^\top \Sigma w$. On cherche, pour chaque niveau de rendement, le portefeuille de risque minimal :

$$\min_w \; w^\top \Sigma w \quad \text{s.c.} \quad w^\top \mu = r^*, \; \sum_i w_i = 1, \; 0 \le w_i \le 15\%$$

La limite de 15 % par action évite les portefeuilles trop concentrés. La covariance est estimée par **Ledoit-Wolf** (shrinkage), plus robuste que la covariance empirique.

In [ ]:
W_MAX = 0.15
rand = markowitz.random_portfolios(mu, sigma, n=5000)
front = markowitz.efficient_frontier(mu, sigma, n_points=30, w_max=W_MAX)
w_mv = markowitz.min_variance(sigma, W_MAX)
w_ms = markowitz.max_sharpe(mu, sigma, w_max=W_MAX)
w_eq = np.full(len(mu), 1 / len(mu))

fig, ax = plt.subplots()
sc = ax.scatter(rand.volatilite, rand.rendement, c=rand.sharpe, cmap="viridis", s=4, alpha=0.5)
plt.colorbar(sc, label="Sharpe")
ax.plot(front.volatilite, front.rendement, "k-", lw=2, label="Frontière efficiente")
for w, lab, m in [(w_mv, "Variance minimale", "o"), (w_ms, "Max Sharpe", "*"), (w_eq, "Équipondéré", "s")]:
    r, v, _ = markowitz.perf(w, mu.values, sigma.values)
    ax.scatter(v, r, marker=m, s=180, edgecolor="k", zorder=5, label=lab)
ax.set_xlabel("Volatilité annuelle"); ax.set_ylabel("Rendement annuel"); ax.legend()
ax.set_title("Portefeuilles aléatoires et frontière efficiente");

In [ ]:
alloc = pd.DataFrame({"Variance minimale": w_mv, "Max Sharpe": w_ms}, index=prices.columns.map(names))
alloc = alloc[(alloc > 0.005).any(axis=1)].sort_values("Max Sharpe")
alloc.plot.barh(figsize=(9, 7), title="Composition des portefeuilles optimaux (poids > 0,5 %)");

⚠️ **Attention** : ces portefeuilles sont calculés avec toute la période, donc avec des informations « du futur ». C'est pour cela qu'ils paraissent excellents. Seul le backtest de la partie 4 dit si la méthode fonctionne vraiment.

## 3. Clustering des actions

On mesure la « distance » entre deux actions par $d_{ij} = \sqrt{(1-\rho_{ij})/2}$ : 0 si elles bougent exactement pareil, 1 si elles sont parfaitement opposées.

**a) Classification ascendante hiérarchique (CAH)**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(Z, labels=[names[t] for t in returns.columns], leaf_rotation=90, leaf_font_size=8, ax=ax, color_threshold=Z[-5, 2])
ax.set_title("Dendrogramme (méthode de Ward)"); ax.grid(False);
for c in sorted(labels_h.unique()):
    print(f"Cluster {c} :", ", ".join(names[t] for t in labels_h.index[labels_h == c]))

**b) K-means sur les composantes principales (ACP)**, pour vérifier que les groupes sont stables d'une méthode à l'autre.

In [ ]:
labels_k, emb = clustering.kmeans_clusters(returns, n_clusters=6)
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(emb[:, 0], emb[:, 1], c=labels_k, cmap="tab10", s=60)
for i, t in enumerate(returns.columns):
    ax.annotate(names[t], emb[i, :2], fontsize=7)
ax.set_xlabel("Composante 1 (≈ facteur marché)"); ax.set_ylabel("Composante 2")
ax.set_title("Actions projetées sur les 2 premières composantes, colorées par cluster K-means")
print(pd.crosstab(labels_h, labels_k, rownames=["CAH"], colnames=["K-means"]))

**c) Allocation à partir des clusters**
- *Parité de risque par clusters* : poids inverse-volatilité dans chaque groupe, puis entre groupes. Chaque groupe pèse à peu près autant dans le risque total.
- *HRP (Hierarchical Risk Parity, López de Prado 2016)* : on coupe l'arbre en deux récursivement et on répartit selon la variance de chaque moitié. Pas besoin d'inverser la matrice de covariance, donc beaucoup plus stable que Markowitz.

In [ ]:
w_crp = clustering.cluster_risk_parity(returns, labels_h, sigma.values)
w_hrp = clustering.hrp(returns, sigma.values)
by_cluster = pd.DataFrame({"Max Sharpe": w_ms, "Parité clusters": w_crp, "HRP": w_hrp, "1/N": w_eq},
                          index=returns.columns).groupby(labels_h).sum()
by_cluster.T.plot.bar(stacked=True, title="Poids total investi dans chaque cluster", rot=0).legend(title="Cluster", bbox_to_anchor=(1, 1));

## 4. Backtest walk-forward

Règles :
- au dernier jour de chaque mois, on estime $\mu$ et $\Sigma$ sur **les 252 jours précédents uniquement** (1 an) ;
- on calcule les nouveaux poids et on paie **10 points de base** (0,10 %) sur le volume échangé ;
- entre deux rebalancements, les poids dérivent avec les prix (comme dans la réalité).

In [ ]:
rets, weights, table = backtest.run_all(returns, bench, lookback=252, cost_bps=10)
nav = (1 + rets).cumprod()
ax = nav.plot(logy=True, title="Valeur de 1 € investi (échelle logarithmique)")
ax.set_ylabel("Valeur");

In [ ]:
table.style.format({c: "{:.1%}" for c in ["Rendement annualisé", "Volatilité", "Perte max (drawdown)", "VaR 95 % (jour)"]}
                  | {c: "{:.2f}" for c in ["Sharpe", "Sortino", "Calmar"]}).background_gradient(subset=["Sharpe"], cmap="Greens")

In [ ]:
dd = nav / nav.cummax() - 1
dd.plot(title="Drawdown : perte depuis le dernier plus haut", alpha=0.8).set_ylabel("Perte");

In [ ]:
# Stabilité des poids : un portefeuille qui change beaucoup coûte cher en frais
turn = {k: w.diff().abs().sum(axis=1).mean() for k, w in weights.items()}
print("Rotation mensuelle moyenne (somme des |variations de poids|) :")
print(pd.Series(turn).sort_values().map("{:.1%}".format).to_string())
w = weights["Max Sharpe"].rename(columns=names)
w.loc[:, w.mean().nlargest(10).index].plot.area(figsize=(11, 5), title="Max Sharpe : évolution des 10 plus gros poids").legend(bbox_to_anchor=(1, 1), fontsize=7);

## 5. Analyse de sensibilité

Un bon résultat ne doit pas dépendre d'un paramètre précis. On fait varier la fenêtre d'estimation.

In [ ]:
sens = {}
for lb in [126, 252, 504]:
    _, _, t = backtest.run_all(returns, None, lookback=lb)
    sens[f"{lb} jours"] = t["Sharpe"]
pd.DataFrame(sens).round(2)

## 6. Conclusions et limites

À compléter avec tes résultats après exécution. Points à commenter :
- Quelle stratégie a le meilleur Sharpe ? La plus faible perte maximale ?
- Le « Max Sharpe » est souvent décevant hors échantillon : les rendements espérés $\mu$ sont très mal estimés sur un an, et l'optimiseur amplifie ces erreurs (« error maximization », Michaud 1989).
- Les méthodes qui n'utilisent que la covariance (variance minimale, HRP, parité par clusters) sont en général plus stables.

**Limites** : biais du survivant (on prend les actions du CAC 40 *d'aujourd'hui*), frais simplifiés, pas de dividendes taxés, pas de contraintes de liquidité.

**Pistes** : modèle de Black-Litterman, covariance GARCH/DCC, contraintes sectorielles, CVaR au lieu de la variance.